# Colab GPU workflow: Phase 5 3D ResNet grid

Orchestration only - every training cell shells out to `scripts/train_resnet_baseline.py` and
`scripts/aggregate_resnet_grid.py` in the cloned repo. No training/model/eval logic lives in
this notebook (CLAUDE.md §6: "no logic in notebooks"). See
`docs/adr/004-colab-training-workflow.md` for why this workflow exists and what each design
choice is for.

**Before running:** Runtime > Change runtime type > GPU (A100 if available on your tier, T4
otherwise - `--amp-dtype` auto-selects fp16 vs bf16 based on what the assigned GPU supports, no
action needed either way).

**Order:** run cells 1-4 once per session. Cell 5 is the 1-fold/1-seed pilot - run it first and
read its output before touching cell 6 (the full grid), per the standing project rule: no full
grid without a completed, timed pilot and explicit go-ahead.

## 1. Mount Drive (persistent storage for checkpoints + the preprocessed cache)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/glioma-biomarker-data'
!mkdir -p "$DRIVE_ROOT/tumor_crop_cache" "$DRIVE_ROOT/experiments"

## 2. Clone the repo and install pinned dependencies

The repo itself carries no MRI data (gitignored) - only code, `splits/`, `metadata/`, and
`configs/`. `torch==2.14.0` is reinstalled from the CUDA wheel index since the pin in
`pyproject.toml` was validated on macOS/MPS, not CUDA - **check the printed CUDA version below
matches an available `cu12x` build before trusting this install; if it doesn't, stop and report
back rather than silently falling back to a CPU wheel.**

In [ ]:
!git clone https://github.com/pranathii-31/glioma-biomarker.git /content/glioma-biomarker
%cd /content/glioma-biomarker
!git rev-parse HEAD

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version 2>/dev/null || echo 'nvcc not found - check CUDA via torch after install instead'

In [ ]:
# Pinned deps from pyproject.toml, minus dev tooling (pre-commit is not needed on Colab -
# results are committed from the local machine after pulling metrics.json back, see cell 9).
!pip install -q torch==2.14.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q monai==1.6.0 nibabel==5.4.2 SimpleITK==2.5.6 numpy==2.2.6 pandas==2.3.3 \
    scipy==1.15.3 scikit-learn==1.7.2 omegaconf==2.3.1

import torch

print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())

## 3. Bring in the preprocessed `tumor_crop` cache

One-time: upload your local `data/processed/tumor_crop_<hash>/` (~3.1GB, 465 cached patients)
to the Drive folder above via Drive's own web upload or `rclone` - **not** through this notebook
and **not** through git. This cell only copies it from Drive into the Colab local disk (faster
repeated I/O during training than reading every batch straight off a Drive mount).

In [ ]:
!mkdir -p data/processed
!cp -r "$DRIVE_ROOT"/tumor_crop_cache/* data/processed/ 2>/dev/null || \
    echo 'Nothing found under $DRIVE_ROOT/tumor_crop_cache - upload the cache there first.'
!ls data/processed/ && find data/processed -name '*.npy' | wc -l

## 4. Point experiment output at Drive (so a session disconnect doesn't lose checkpoints)

In [ ]:
!rm -rf experiments
!ln -s "$DRIVE_ROOT/experiments" experiments
!ls -la experiments

## 5. Pilot: 1 fold, 1 seed, capped epochs

**Stop and read this cell's output before running anything below it.** Report the actual
seconds/epoch and peak GPU memory from the printed summary / `experiments/resnet18_fold_0_seed0/metrics.json`
back before starting the full grid - this is the same gate the M4 pilot was meant to clear and
didn't.

In [ ]:
!python scripts/train_resnet_baseline.py \
    --architecture resnet18 --fold fold_0 --seed 0 --max-epochs 5

If this cell's Colab session disconnects mid-run, rerun the same command with `--resume` added
- it will pick up from the latest checkpoint in `experiments/resnet18_fold_0_seed0/checkpoints/`
(on Drive, so it survives the disconnect) rather than restarting.

## 6. Full grid (only after the pilot is reviewed and you have explicit go-ahead)

5 folds x 3 seeds x {resnet18, resnet34} = 30 runs. Each cell below trains one
(architecture, fold, seed) combination. `--epoch-limit` caps a single invocation to a time
budget appropriate for one Colab session; rerun the same command with `--resume` appended in a
later session to continue. Adjust `EPOCH_LIMIT` to whatever one session can realistically cover
(unknown until the pilot's seconds/epoch is in hand - fill this in after step 5).

In [ ]:
import subprocess

ARCHITECTURES = ['resnet18', 'resnet34']
FOLDS = ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
SEEDS = [0, 1, 2]
EPOCH_LIMIT = None  # set from the pilot's timing before running this for real

for architecture in ARCHITECTURES:
    for fold in FOLDS:
        for seed in SEEDS:
            cmd = [
                'python', 'scripts/train_resnet_baseline.py',
                '--architecture', architecture, '--fold', fold, '--seed', str(seed),
                '--resume',
            ]
            if EPOCH_LIMIT is not None:
                cmd += ['--epoch-limit', str(EPOCH_LIMIT)]
            print('>>>', ' '.join(cmd))
            subprocess.run(cmd, check=True)

## 7. Aggregate each (architecture, seed) across its 5 folds into a reportable CV result

Only run this once all 5 folds for a given (architecture, seed) have actually finished (not
mid-budget checkpoints) - `scripts/aggregate_resnet_grid.py` raises with a clear message
naming which folds are missing/incomplete otherwise.

In [ ]:
for architecture in ARCHITECTURES:
    for seed in SEEDS:
        subprocess.run(
            ['python', 'scripts/aggregate_resnet_grid.py',
             '--architecture', architecture, '--seed', str(seed)],
            check=True,
        )

## 8. Regenerate `results/baselines.md`

In [ ]:
!python scripts/build_results_table.py
!cat results/baselines.md

## 9. Bring results back to the local repo

`experiments/` here is a symlink into Drive, not a git-tracked copy - **do not `git add` from
inside this notebook.** On your local machine: sync the aggregated `experiments/*/metrics.json`
(+ `config.yaml`/`git_commit.txt`/`splits_hash.txt`/`env.txt`, never `checkpoints/`) down from
Drive into the local repo's `experiments/`, rerun `make lint && make test`, regenerate
`results/baselines.md`, and commit from there - keeps commit authorship, pre-commit hooks, and
the git safety protocol all on the machine that actually has them configured.